# Preventra CMS Readmission Model — Phase 1 (Discharge-Time Risk)

This notebook trains the **Phase 1** discharge-time readmission model — see
`docs/Preventra CMS Feasibility 4Week Plan.docx` and `docs/cms/cms_migration_guide.md` for the
migration context. It is scoped to Phase 1 only: the source data here
(`data/cms/finalmerged.csv`) is a 200-row extract already merged and partially engineered
across all 20 CMS DE-SynPUF samples (see its `SOURCE_FILE` column), not the raw per-sample
claims tables — Phase 2's weekly post-discharge features need raw Carrier/Outpatient/PDE
claims that this extract doesn't carry, so that track isn't built here.

**What's already done in `finalmerged.csv`** (built upstream, not by this notebook):
- The inpatient discharge spine, joined to beneficiary demographics and chronic-condition flags
- The `READMITTED_30D` label and `DAYS_TO_NEXT_ADMISSION`, derived from each patient's next admission
- `PRIOR_INPATIENT_STAYS_L1Y`, `PRIOR_OUTPATIENT_VISIT_COUNT`, `PRIOR_OUTPATIENT_PAYMENT_SUM` — prior-utilization lookbacks

**What this notebook still does** (the diagnosis/demographic columns are raw, not yet
model-ready):
- Charlson Comorbidity Index from the raw `ICD9_DGNS_CD_*` columns
- Behavioral-health flag from the same diagnosis codes
- Frailty proxy, one-hot race, total chronic-condition count, discharge-complexity flag
- Assembling the final feature matrix and training/evaluating the Decision Tree model

**Not available in this extract** (so not modeled here): Part D pharmacy fills and
Carrier-claims-based ED/ambulance/PCP-follow-up signals — this dataset only carries
Inpatient + Beneficiary + a prior-outpatient-utilization summary, not the full claims tables
Phase 1's original design called for. If those get added to a future extract, the feature
set here should grow to match; nothing about the code below assumes they're absent.

In [ ]:
import re

import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", 60)

RANDOM_STATE = 42
DATA_PATH = "finalmerged.csv"  # expected alongside this notebook, in Notebooks/

## 1. Load `finalmerged.csv`

Dates here are already ISO `YYYY-MM-DD` strings (unlike the raw CMS `YYYYMMDD` integer
format the multi-sample loader had to special-case), so a plain `pd.to_datetime` is enough.

In [ ]:
df = pd.read_csv(DATA_PATH)
df["CLM_ADMSN_DT"] = pd.to_datetime(df["CLM_ADMSN_DT"])
df["NCH_BENE_DSCHRG_DT"] = pd.to_datetime(df["NCH_BENE_DSCHRG_DT"])

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Sample folders represented: {df['SOURCE_FILE'].nunique()} "
      f"({sorted(df['SOURCE_FILE'].unique())[:5]}...)")
print(f"Positive rate (READMITTED_30D): {df['READMITTED_30D'].mean():.2%} "
      f"({df['READMITTED_30D'].sum()} positive of {len(df)})")

null_counts = df.isnull().sum()
print(f"\nColumns with missing values:\n{null_counts[null_counts > 0]}")

## 2. Clean the diagnosis/procedure code columns

Pandas infers a dtype per column independently: a diagnosis column where every non-null
value happens to be all-digit numbers gets read as `float64` (e.g. `ICD9_DGNS_CD_10` and
`ICD9_PRCDR_CD_1` in this file), while a column with even one alphanumeric code (like a
`V`-prefixed code) stays `object`/string. This causes two problems downstream:

1. **Silent scoring bug**: the CCI weight lookup only matches a code if it's a Python `str`
   (`isinstance(code, str)`) — a `float64` column would silently score a comorbidity weight
   of **0 for every code in it**, with no error.
2. **Value corruption if naively stringified**: `str(4019.0)` is `"4019.0"`, not `"4019"` —
   stripping the `.` later would turn it into `"40190"`, a different code entirely.

`clean_diag_code_column` fixes both by routing numeric-dtype code columns through a nullable
integer first (`Int64`), which strips the trailing `.0` cleanly, before converting to string.

In [ ]:
def clean_diag_code_column(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return s.astype("Int64").astype(str).replace("<NA>", pd.NA)
    return s


DGNS_COLS = [c for c in df.columns if c.startswith("ICD9_DGNS_CD_")]
PRCDR_COLS = [c for c in df.columns if c.startswith("ICD9_PRCDR_CD_")]

for col in DGNS_COLS + PRCDR_COLS:
    df[col] = clean_diag_code_column(df[col])

print(f"Diagnosis columns: {DGNS_COLS}")
print(f"Procedure columns: {PRCDR_COLS}")

## 3. Charlson Comorbidity Index + behavioral health flag

Same weight table as `features/cci.py` (kept identical so scores stay comparable to the
existing diabetes-data baseline), applied across however many `ICD9_DGNS_CD_*` columns this
extract carries.

In [ ]:
CCI_MAP: list[tuple[str, int]] = [
    (r"^410|^412",                          1),  # Myocardial infarction
    (r"^428",                               1),  # Congestive heart failure
    (r"^4[45]",                             1),  # Peripheral vascular disease
    (r"^43[0-8]",                           1),  # Cerebrovascular disease
    (r"^290",                               1),  # Dementia
    (r"^49[0-6]|^500|^505|^5064",          1),  # Chronic pulmonary disease
    (r"^710[01]|^7140|^7141|^7142|^7148",  1),  # Connective tissue disease
    (r"^53[1-4]",                           1),  # Peptic ulcer disease
    (r"^571",                               1),  # Mild liver disease
    (r"^250[0-3]",                          1),  # Diabetes without complications
    (r"^196|^197|^198|^199",               6),  # Metastatic solid tumor
    (r"^042|^043|^044",                    6),  # AIDS/HIV
    (r"^572[2-8]",                          3),  # Moderate/severe liver disease
    (r"^342|^344[01]",                      2),  # Hemiplegia
    (r"^58[2-3]|^585|^586|^5880",          2),  # Moderate/severe renal disease
    (r"^250[4-9]",                          2),  # Diabetes with end-organ damage
    (r"^1[4-9][0-9]|^20[0-8]",            2),  # Tumor (non-metastatic)
    (r"^204[1]|^205[3]|^206[3]|^207[12]", 2),  # Leukemia
    (r"^200|^201|^202",                    2),  # Lymphoma
]
_COMPILED_CCI = [(re.compile(p), w) for p, w in CCI_MAP]
_BH_LOW, _BH_HIGH = 291, 319  # ICD-9 range for mental/behavioural health disorders


def _icd9_to_cci_weight(code) -> int:
    if not isinstance(code, str) or not code.strip():
        return 0
    code_clean = code.strip().upper().replace(".", "")
    for pattern, weight in _COMPILED_CCI:
        if pattern.match(code_clean):
            return weight
    return 0


def compute_cci_row(row: pd.Series, dgns_cols: list) -> int:
    seen_weights = set()
    total = 0
    for col in dgns_cols:
        w = _icd9_to_cci_weight(row.get(col))
        if w > 0 and w not in seen_weights:
            total += w
            seen_weights.add(w)
    return total


def has_behavioral_health_code(row: pd.Series, dgns_cols: list) -> int:
    for col in dgns_cols:
        code = str(row.get(col, "")).strip().split(".")[0]
        try:
            if _BH_LOW <= int(code) <= _BH_HIGH:
                return 1
        except ValueError:
            continue
    return 0


df["charlson_comorbidity_index"] = df.apply(lambda r: compute_cci_row(r, DGNS_COLS), axis=1)
df["behavioral_health_flag"] = df.apply(lambda r: has_behavioral_health_code(r, DGNS_COLS), axis=1)

print(f"CCI distribution:\n{df['charlson_comorbidity_index'].value_counts().sort_index()}")
print(f"\nBehavioral health flag rate: {df['behavioral_health_flag'].mean():.2%}")

## 4. Demographics, chronic-condition totals, and discharge complexity

Two rows in this extract have no demographic/chronic-condition data at all (a beneficiary
summary that didn't match) — missing chronic-condition flags are filled with 0 (conservative:
"unknown" treated as "not present", consistent with `docs/diabetic/feature_notes.md`'s null-handling
convention), and missing age is filled with the column median rather than 0 so it doesn't
corrupt the frailty threshold.

In [ ]:
SP_COLS = ["SP_ALZHDMTA", "SP_CHF", "SP_CHRNKIDN", "SP_CNCR", "SP_COPD", "SP_DEPRESSN",
           "SP_DIABETES", "SP_ISCHMCHT", "SP_OSTEOPRS", "SP_RA_OA", "SP_STRKETIA"]
# CMS DE-SynPUF codebook: 1=White, 2=Black, 3=Others, 5=Hispanic.
RACE_CODES = [1, 2, 3, 5]

df["age_at_admission"] = df["AGE_AT_ADMISSION"].fillna(df["AGE_AT_ADMISSION"].median())
df["frailty_proxy"] = (df["age_at_admission"] >= 75).astype(int)

df["sex_male"] = (df["BENE_SEX_IDENT_CD"] == 1).astype(int)

for col in SP_COLS:
    df[col] = df[col].fillna(0).astype(int)
df["total_chronic_conditions"] = df[SP_COLS].sum(axis=1)

# Fixed categories (not discovered from the data) so race columns stay well-defined even if
# a category is entirely absent from a given slice of the data.
race_cat = pd.Categorical(df["BENE_RACE_CD"], categories=RACE_CODES)
race_dummies = pd.get_dummies(race_cat, prefix="race_cd", dtype=int)
race_dummies["race_cd_other"] = (~df["BENE_RACE_CD"].isin(RACE_CODES)).astype(int)
df = pd.concat([df, race_dummies], axis=1)
RACE_DUMMY_COLS = [f"race_cd_{c}" for c in RACE_CODES] + ["race_cd_other"]

df["diagnosis_code_count"] = df[DGNS_COLS].notna().sum(axis=1)
df["procedure_code_count"] = df[PRCDR_COLS].notna().sum(axis=1)
df["complex_discharge_flag"] = (
    (df["LENGTH_OF_STAY"] > 7)
    & (df["diagnosis_code_count"] > 7)
    & (df["procedure_code_count"] > 2)
).astype(int)

df["is_zero_ip_deductible"] = (df["NCH_BENE_IP_DDCTBL_AMT"].fillna(0) == 0).astype(int)
df["high_utilizer_flag"] = (df["PRIOR_INPATIENT_STAYS_L1Y"] >= 2).astype(int)

print(f"Frailty rate: {df['frailty_proxy'].mean():.2%} | "
      f"Complex discharge rate: {df['complex_discharge_flag'].mean():.2%} | "
      f"High utilizer rate: {df['high_utilizer_flag'].mean():.2%}")

## 5. Assemble the final Phase 1 feature matrix

`LENGTH_OF_STAY`, `CLM_PMT_AMT` (this admission's own claim payment — a severity/cost proxy
distinct from the prior-year spend used in the earlier raw-claims design), `CLM_DRG_CD`, and
the three `PRIOR_*` lookback columns are used directly since they're already computed
upstream. `CLM_DRG_CD` is kept as a plain numeric code rather than one-hot encoded — at 200
rows it's too high-cardinality to one-hot usefully, and a Decision Tree can still split on it
directly.

In [ ]:
LABEL_COL = "READMITTED_30D"
METADATA_COLS = [
    "DESYNPUF_ID", "CLM_ID", "CLM_ADMSN_DT", "NCH_BENE_DSCHRG_DT",
    "DAYS_TO_NEXT_ADMISSION", "SOURCE_FILE",
]
PHASE1_FEATURE_COLS = [
    "age_at_admission", "frailty_proxy", "sex_male",
    "total_chronic_conditions", *SP_COLS, *RACE_DUMMY_COLS,
    "LENGTH_OF_STAY", "CLM_PMT_AMT", "CLM_DRG_CD",
    "charlson_comorbidity_index", "behavioral_health_flag",
    "diagnosis_code_count", "procedure_code_count", "complex_discharge_flag",
    "is_zero_ip_deductible",
    "PRIOR_INPATIENT_STAYS_L1Y", "PRIOR_OUTPATIENT_VISIT_COUNT",
    "PRIOR_OUTPATIENT_PAYMENT_SUM", "high_utilizer_flag",
]

phase1_df = df[METADATA_COLS + PHASE1_FEATURE_COLS + [LABEL_COL]].copy()
phase1_df[PHASE1_FEATURE_COLS] = phase1_df[PHASE1_FEATURE_COLS].fillna(0)

null_counts = phase1_df[PHASE1_FEATURE_COLS + [LABEL_COL]].isnull().sum()
assert null_counts.sum() == 0, f"Unexpected nulls in Phase 1 features:\n{null_counts[null_counts > 0]}"

print(f"Final feature matrix: {phase1_df.shape[0]} rows x {len(PHASE1_FEATURE_COLS)} features")
print(f"Positive rate: {phase1_df[LABEL_COL].mean():.2%}")

# phase1_df.to_csv("phase1_features_final.csv", index=False)  # uncomment when ready to persist
phase1_df.head()

## 6. Train the discharge-time model

Same conventions as the existing baseline pipeline (`models/train.py`): a temporal
train/test split (earliest admissions train, latest test), a `DecisionTreeClassifier` with
`class_weight="balanced"`, then a guarded `GridSearchCV` tuning pass. With 18 total positives
across 200 rows, an 80/20 split leaves enough positives in the training fold for
cross-validation to actually run (unlike the very first ~100-row smoke-test data this
pipeline was originally prototyped against).

In [ ]:
def temporal_split(data: pd.DataFrame, date_col: str, ratio: float = 0.8):
    data = data.sort_values(date_col).reset_index(drop=True)
    split_idx = int(len(data) * ratio)
    return data.iloc[:split_idx].copy(), data.iloc[split_idx:].copy()


def safe_auc(y_true, y_prob):
    '''roc_auc_score requires both classes present in y_true; return NaN otherwise.'''
    if y_true.nunique() < 2:
        return float("nan")
    return roc_auc_score(y_true, y_prob)


def evaluate_classifier(name, model, X_test, y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    auc = safe_auc(y_test, y_prob)
    print(f"--- {name} ---")
    print(f"AUC-ROC   : {auc:.4f}" if pd.notna(auc) else "AUC-ROC   : n/a (single class in test set)")
    print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"Confusion matrix:\n{confusion_matrix(y_test, y_pred)}")
    print(classification_report(y_test, y_pred, zero_division=0))
    return y_prob


train_df, test_df = temporal_split(phase1_df, "CLM_ADMSN_DT", ratio=0.8)
X_train, y_train = train_df[PHASE1_FEATURE_COLS], train_df[LABEL_COL]
X_test, y_test = test_df[PHASE1_FEATURE_COLS], test_df[LABEL_COL]

print(f"Split: train={len(train_df)} ({y_train.mean():.2%} positive) | "
      f"test={len(test_df)} ({y_test.mean():.2%} positive)")

baseline_model = DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)
baseline_model.fit(X_train, y_train)
_ = evaluate_classifier("Phase 1 baseline DecisionTree", baseline_model, X_test, y_test)

In [ ]:
PARAM_GRID = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_leaf": [5, 10, 20],
    "min_samples_split": [10, 20],
}

n_pos_train = int(y_train.sum())
cv_folds = min(5, n_pos_train)

if cv_folds >= 2:
    grid = GridSearchCV(
        DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        PARAM_GRID, scoring="roc_auc", cv=cv_folds, n_jobs=-1, refit=True,
    )
    grid.fit(X_train, y_train)
    phase1_model = grid.best_estimator_
    print(f"Best params: {grid.best_params_} | Best CV AUC: {grid.best_score_:.4f}")
    _ = evaluate_classifier("Phase 1 tuned DecisionTree", phase1_model, X_test, y_test)
else:
    phase1_model = baseline_model
    print(f"Tuning skipped -- only {n_pos_train} positive example(s) in the training split, "
          f"not enough to cross-validate.")

feature_importance = (
    pd.Series(phase1_model.feature_importances_, index=PHASE1_FEATURE_COLS)
    .sort_values(ascending=False)
)
print("\nTop 10 feature importances:")
print(feature_importance.head(10))

## 7. Summary

- 200 admissions across all 20 CMS DE-SynPUF samples, 18 readmitted within 30 days (9%).
- Feature set: demographics + 11 chronic-condition flags, Charlson Comorbidity Index,
  behavioral-health flag, discharge complexity (LOS/diagnosis/procedure counts), this
  admission's own claim payment and DRG, and the pre-computed prior-utilization lookbacks
  (`PRIOR_INPATIENT_STAYS_L1Y`, `PRIOR_OUTPATIENT_VISIT_COUNT`, `PRIOR_OUTPATIENT_PAYMENT_SUM`).
- Trained with a temporally-split, class-balanced Decision Tree, tuned via `GridSearchCV`
  where the training fold has enough positives to cross-validate.
- **Gaps versus the original Phase 1 design** (per the feasibility doc's data-source list):
  no Part D pharmacy adherence signal and no Carrier-claims-based ED/ambulance/PCP-follow-up
  signal are present in this extract, since it only carries Inpatient + Beneficiary +
  a prior-outpatient-utilization summary. If a future extract adds those tables back in,
  the feature set here should be extended to match — Phase 2's weekly re-scoring track in
  particular needs the raw claims tables this extract doesn't carry.